## Creación de Tensores

In [1]:
"""
Preprocesamiento del conjunto de datos de anestesia.

Este script lee los archivos CSV de cada conjunto (entrenamiento, validación
y prueba), genera las ventanas deslizantes, las normaliza con estadísticas
globales calculadas sobre el conjunto de entrenamiento y guarda el resultado
como tensores de PyTorch (.pt) listos para cargarse en la GPU durante el
entrenamiento.

Criterios de construcción (implementados en Normalizacion_BIS.py):
    * Normalización global: cada señal se estandariza con la media y la
      desviación estándar del conjunto de entrenamiento, lo que conserva la
      magnitud de las infusiones y el nivel del BIS.
    * Relleno: las infusiones se rellenan con cero y el BIS con el primer
      valor del registro, en unidades originales y antes de normalizar.
    * Alineación temporal: la ventana k abarca las muestras k-W+1, ..., k y
      su etiqueta es BIS[k+1]; un registro de T muestras produce T-1 pares.

Estructura de salida por conjunto:
    infusiones   : torch.float16  (N, 2, W)
    bis_hist     : torch.float16  (N, W)
    bis_hist_gt  : torch.float32  (N, W)   -- en unidades de BIS, para la pérdida auxiliar
    covariables  : torch.float32  (N, 4)
    etiquetas    : torch.float32  (N,)     -- BIS en el instante siguiente
    estadisticas : dict                    -- parámetros de normalización empleados

Las estadísticas se guardan además en RUTA_ESTADISTICAS, archivo que emplean
las libretas de evaluación y de simulación en lazo cerrado.

Se usa float16 para las entradas normalizadas con el fin de reducir el uso
de memoria y acelerar la transferencia CPU → GPU.
"""

# =============================================================================
# CONFIGURACIÓN
# =============================================================================
TAMANIYO_VENTANA   = 180

RUTA_ENTRENAMIENTO = r"C:\Users\yordi\Desktop\Entrenamiento_Limpios"
RUTA_VALIDACION    = r"C:\Users\yordi\Desktop\Validacion_Limpios"
RUTA_PRUEBA        = r"C:\Users\yordi\Desktop\Prueba_Limpios"

# Archivos de salida
SALIDA_ENTRENAMIENTO = r"C:\Users\yordi\Desktop\dataset_entrenamiento.pt"
SALIDA_VALIDACION    = r"C:\Users\yordi\Desktop\dataset_validacion.pt"
SALIDA_PRUEBA        = r"C:\Users\yordi\Desktop\dataset_prueba.pt"

# Carpeta que contiene Normalizacion_BIS.py
CARPETA_LIBRETAS   = r"C:\Users\yordi\Desktop\Libretas Python"


import os
import sys
import glob
import time
import numpy as np
import pandas as pd
import torch

if CARPETA_LIBRETAS not in sys.path:
    sys.path.insert(0, CARPETA_LIBRETAS)
from Normalizacion_BIS import (RUTA_ESTADISTICAS, calcular_estadisticas,
                               guardar_estadisticas, describir_estadisticas,
                               construir_ventanas_paciente)


# =============================================================================
# FUNCIÓN PRINCIPAL DE CONVERSIÓN
# =============================================================================
def listar_csv(ruta_carpeta: str) -> list:
    """Devuelve la lista ordenada de archivos CSV de una carpeta."""
    archivos = sorted(glob.glob(os.path.join(ruta_carpeta, "*.csv")))
    if not archivos:
        raise FileNotFoundError(
            f"No se encontraron archivos CSV en: {ruta_carpeta}"
        )
    return archivos


def procesar_carpeta(ruta_carpeta: str, nombre_conjunto: str,
                     estadisticas: dict) -> dict:
    """
    Lee todos los archivos CSV de una carpeta, genera las ventanas
    deslizantes normalizadas y devuelve un diccionario de tensores.

    Parámetros
    ----------
    ruta_carpeta    : ruta al directorio con archivos CSV
    nombre_conjunto : nombre del conjunto, para los mensajes de la consola
    estadisticas    : estadísticas globales de normalización

    Retorna
    -------
    Diccionario con las claves:
        'infusiones'   : (N, 2, W)  float16
        'bis_hist'     : (N, W)     float16
        'bis_hist_gt'  : (N, W)     float32
        'covariables'  : (N, 4)     float32
        'etiquetas'    : (N,)       float32
        'estadisticas' : dict
    """
    archivos = listar_csv(ruta_carpeta)
    print(f"\n[{nombre_conjunto.upper()}] {len(archivos)} archivos encontrados.")

    # Listas que acumulan los datos de todos los pacientes
    lista_infusiones  = []
    lista_bis_hist    = []
    lista_bis_hist_gt = []
    lista_covariables = []
    lista_etiquetas   = []

    W = TAMANIYO_VENTANA

    for idx_archivo, ruta in enumerate(archivos):
        df = pd.read_csv(ruta, usecols=[
            "PPF20_RATE", "RFTN20_RATE", "BIS",
            "Edad_Norm", "Peso_norm", "Altura_norm", "Sexo_norm"
        ])

        covariables = np.array([
            df["Edad_Norm"].iloc[0],
            df["Peso_norm"].iloc[0],
            df["Altura_norm"].iloc[0],
            df["Sexo_norm"].iloc[0],
        ], dtype=np.float32)

        ventanas = construir_ventanas_paciente(
            df["PPF20_RATE"].to_numpy(dtype=np.float32),
            df["RFTN20_RATE"].to_numpy(dtype=np.float32),
            df["BIS"].to_numpy(dtype=np.float32),
            W, estadisticas,
        )
        if ventanas is None:
            print(f"  Archivo [{idx_archivo + 1:3d}/{len(archivos)}]: "
                  f"{os.path.basename(ruta):40s} | omitido (menos de dos muestras)")
            continue

        n_pares = len(ventanas["etiquetas"])
        lista_infusiones.append(ventanas["infusiones"])
        lista_bis_hist.append(ventanas["bis_hist"])
        lista_bis_hist_gt.append(ventanas["bis_hist_gt"])
        lista_covariables.append(np.tile(covariables, (n_pares, 1)))   # (T-1, 4)
        lista_etiquetas.append(ventanas["etiquetas"])

        # Reporte de progreso por archivo
        print(
            f"  Archivo [{idx_archivo + 1:3d}/{len(archivos)}]: "
            f"{os.path.basename(ruta):40s} | "
            f"{n_pares + 1:5d} pasos → {n_pares} pares entrada-etiqueta"
        )

    # Concatenación de todos los pacientes en un único tensor
    print(f"\n  Concatenando arreglos del conjunto {nombre_conjunto}...")
    t0 = time.time()

    inf_total  = np.concatenate(lista_infusiones,  axis=0)   # (N, 2, W)
    bh_total   = np.concatenate(lista_bis_hist,    axis=0)   # (N, W)
    bh_gt_tot  = np.concatenate(lista_bis_hist_gt, axis=0)   # (N, W)
    cov_total  = np.concatenate(lista_covariables, axis=0)   # (N, 4)
    eta_total  = np.concatenate(lista_etiquetas,   axis=0)   # (N,)

    print(
        f"  Concatenación completada en {time.time() - t0:.1f} s. "
        f"Total de muestras: {len(eta_total):,}"
    )

    # Conversión a tensores de PyTorch
    dataset = {
        "infusiones":   torch.from_numpy(inf_total),
        "bis_hist":     torch.from_numpy(bh_total),
        "bis_hist_gt":  torch.from_numpy(bh_gt_tot),
        "covariables":  torch.from_numpy(cov_total),
        "etiquetas":    torch.from_numpy(eta_total),
        "estadisticas": estadisticas,
    }

    # Reporte de memoria
    mb_total = sum(v.nbytes for v in dataset.values()
                   if isinstance(v, torch.Tensor)) / (1024 ** 2)
    print(f"  Memoria aproximada del conjunto: {mb_total:.1f} MB")

    return dataset


# =============================================================================
# PUNTO DE ENTRADA
# =============================================================================
if __name__ == "__main__":
    tiempo_inicio_global = time.time()

    # Las estadísticas de normalización se calculan exclusivamente con el
    # conjunto de entrenamiento y se aplican sin cambios a los tres conjuntos
    print("[INFO] Calculando las estadísticas globales de normalización "
          "con el conjunto de entrenamiento...")
    estadisticas = calcular_estadisticas(listar_csv(RUTA_ENTRENAMIENTO))
    guardar_estadisticas(estadisticas, RUTA_ESTADISTICAS)
    print(describir_estadisticas(estadisticas))
    print(f"[INFO] Estadísticas guardadas en: {RUTA_ESTADISTICAS}")

    conjuntos = [
        (RUTA_ENTRENAMIENTO, SALIDA_ENTRENAMIENTO, "entrenamiento"),
        (RUTA_VALIDACION,    SALIDA_VALIDACION,    "validacion"),
        (RUTA_PRUEBA,        SALIDA_PRUEBA,        "prueba"),
    ]

    for ruta_entrada, ruta_salida, nombre in conjuntos:
        t0 = time.time()
        dataset = procesar_carpeta(ruta_entrada, nombre, estadisticas)

        print(f"\n  Guardando en: {ruta_salida}")
        torch.save(dataset, ruta_salida)
        print(
            f"  Guardado completado en {time.time() - t0:.1f} s."
        )

    print(
        f"\n[INFO] Preprocesamiento total finalizado en "
        f"{time.time() - tiempo_inicio_global:.1f} s."
    )
    print("[INFO] Los archivos .pt están listos para el entrenamiento.")


[INFO] Calculando las estadísticas globales de normalización con el conjunto de entrenamiento...
  propofol      : media = 0.00512937, desviación estándar = 0.00783327
  remifentanilo : media = 0.00602943, desviación estándar = 0.00732605
  bis           : media = 44.9932, desviación estándar = 12.6622
[INFO] Estadísticas guardadas en: C:\Users\yordi\Desktop\estadisticas_normalizacion.json

[ENTRENAMIENTO] 480 archivos encontrados.
  Archivo [  1/480]: Paciente_1001_limpio_ConFiltro.csv       |   856 pasos → 855 pares entrada-etiqueta
  Archivo [  2/480]: Paciente_1036_limpio_ConFiltro.csv       |   651 pasos → 650 pares entrada-etiqueta
  Archivo [  3/480]: Paciente_1043_limpio_ConFiltro.csv       |   644 pasos → 643 pares entrada-etiqueta
  Archivo [  4/480]: Paciente_1047_limpio_ConFiltro.csv       |  1343 pasos → 1342 pares entrada-etiqueta
  Archivo [  5/480]: Paciente_104_limpio_ConFiltro.csv        |  1262 pasos → 1261 pares entrada-etiqueta
  Archivo [  6/480]: Paciente_1073_li